In [ ]:
from matplotlib_inline.backend_inline import set_matplotlib_formats
import matplotlib.pyplot as plt
from os import listdir as ls
from IPython.display import display, Markdown
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

from emu_renewal.inputs import get_world_shp
from emu_renewal.constants import OUTPUTS_PATH, MOB_COLOURS
from emu_renewal.outputs import get_param_vals_by_analysis, get_prop_improve
from emu_renewal.plotting import ANALYSIS_NAMES, plot_world_country_outline, plot_prop_improve
from emu_renewal.utils import get_countries_by_continent

set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "46090067"
all_countries = ls(job_path)
countries_by_cont = get_countries_by_continent(all_countries)
disp_posts = {c: get_param_vals_by_analysis("dispersion_proc", job_path / c) for c in all_countries}

In [ ]:
world = get_world_shp()
world["geometry"] = world.simplify(tolerance=0.1)
plt.style.use("default")

fig, axes = plt.subplots(2, 2, figsize=(20, 8), constrained_layout=True)
flat_axes = axes.ravel()

# Strength of evidence for each mobility type
for a, analysis in enumerate(list(ANALYSIS_NAMES.keys())[1:]):
    prop_improve = get_prop_improve(disp_posts, analysis)
    world["prop_improve"] = world["ISO_A3"].map(prop_improve)
    mob_avail = world[world["prop_improve"].notna()]
    mob_unavail = world[world["prop_improve"].isna()]
    
    ax = flat_axes[a]
    mob_avail.plot(column="prop_improve", ax=ax, cmap="coolwarm", legend=True, vmin=0.0, vmax=1.0)
    mob_unavail.plot(ax=ax, color="w", hatch="///", alpha=0.04)
    ax.set_title(analysis)

# Best mobility
best_mob = {c: disp_posts[c].mean().idxmin() for c in all_countries}
world["best_mob"] = world["ISO_A3"].map(best_mob)
world["best_mob_colour"] = world["best_mob"].map(MOB_COLOURS | {"no_mob": "0.45"})
mob_avail = world[world["best_mob_colour"].notna()]
mob_unavail = world[world["best_mob_colour"].isna()]

ax = flat_axes[-1]

# Dummy colour bar to get axis in right position with constrained layout
sm = ScalarMappable(norm=Normalize(vmin=0, vmax=1))
cb = fig.colorbar(sm, ax=ax)
cb.ax.set_visible(False)

mob_avail.plot(ax=ax, color=mob_avail["best_mob_colour"])
mob_unavail.plot(ax=ax, color="w", hatch="///", alpha=0.04)

for ax in flat_axes:
    world.boundary.plot(ax=ax, color="black", linewidth=0.2)
    ax.set_xticks([])
    ax.set_yticks([])